# Qwen3-8B: Direct versus explicit CoT, with two answer formats

Run the experiment twice using the same setup and measurement cells: first set
`OUTPUT_FORMAT = "plain"`, then set `OUTPUT_FORMAT = "prefixed"` and rerun
the format, measurement, and experiment cells. The two runs write CSV files
to separate `plain/` and `prefixed/` directories. Run the final plotting cell
after both runs. It reads those CSV files without regenerating model outputs.

The only figures are source-layer profiles at the final answer output:
one panel for the plain response and three panels for the prefixed response,
separately for intermediate $Z$ and answer $Y$. Each panel compares Direct
and CoT with pointwise 95% problem-bootstrap intervals. The plot checks the
saved next-token alignment before naming the three prefixed positions.
The $Y$ readout targets its first digit-bearing answer token. For multi-token
answers this is not a readout of the entire numeral.


In [ ]:
from pathlib import Path
import re, json, gc
from collections import Counter
from decimal import Decimal, InvalidOperation
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM
import jlens

ROOT = Path(r"D:\Researches\InterpreteJLens")
MODEL_DIR = ROOT / "base_models" / "Qwen3-8B"
OUT = ROOT / "jlens_outputs" / "qwen3_8b"
DATA_FILE = ROOT / "data" / "gsm_valid-gold-reasoning-trace_test.json"
LENS_FILE = OUT / "qwen3_8b_wikitext_1000_lens.pt"  # adjust if fit used fewer prompts
MAX_SCAN = None  # None = all 1194; pilot with 10–50 first
MAX_NEW_TOKENS = 1024  # thinking can be long; log truncated generations
LOOKBACK = 8
MIN_WRONG = 4
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
REQUIRE_Z_WRITTEN_IN_COT = True
for path in (MODEL_DIR, DATA_FILE, LENS_FILE):
    if not path.exists(): raise FileNotFoundError(path)
if not torch.cuda.is_available(): raise RuntimeError("CUDA GPU required")
mem_gb=torch.cuda.get_device_properties(0).total_memory/2**30
if mem_gb < 40: raise RuntimeError(f"{mem_gb:.1f} GiB GPU; use a >=48 GB GPU for BF16 8B + readouts")
OUT.mkdir(parents=True,exist_ok=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL_DIR)
model=AutoModelForCausalLM.from_pretrained(MODEL_DIR,torch_dtype=torch.bfloat16,attn_implementation="sdpa").to("cuda").eval()
lens_model=jlens.from_hf(model,tokenizer,compile=False)
lens=jlens.JacobianLens.load(str(LENS_FILE))
layers=sorted(int(x) for x in lens.source_layers)
assert lens.d_model==lens_model.d_model and layers==list(range(lens_model.n_layers-1))
with open(DATA_FILE,encoding="utf-8") as f: data=json.load(f)
print("Model",lens_model,"lens prompts",lens.n_prompts,"source layers",len(layers),"data",len(data))


In [ ]:
# Output parsing and matched targets.
NUMBER_RE=re.compile(r"(?<![\d.])\d+(?:\.\d+)?(?![\d.])")
def canonical(value):
    try:
        v=Decimal(str(value).replace(",", "").strip())
        return format(v.normalize(),"f") if v.is_finite() else None
    except (InvalidOperation,ValueError): return None

def token_id(number):
    ids=tokenizer.encode(str(number),add_special_tokens=False)
    return ids[0] if len(ids)==1 else None

def gold_z_values(sample):
    # The GSM8K-Aug 'steps' list is the gold solution, not the generated CoT.
    steps=sample.get("steps",[])
    vals=[]
    for step in steps[:-1]:
        match=re.search(r"<<[^<>]*=\s*(\d+)\s*>>",step)
        if match:
            z=canonical(match.group(1))
            if z is not None and z!=canonical(sample["answer"]) and token_id(z) is not None:
                vals.append(z)
    return list(dict.fromkeys(vals))

def wrong_controls(z, forbidden):
    if not z.isdigit() or str(int(z)) != z or len(z)>3: return []
    lo=0 if len(z)==1 else 10**(len(z)-1)
    hi=10**len(z)
    return [str(n) for n in range(lo,hi) if str(n) not in forbidden
            and str(n)!=z and token_id(str(n)) is not None]


## Run the experiment once per format
Set `OUTPUT_FORMAT` to `"plain"` and run cells 4–6; then set it to `"prefixed"` and rerun cells 4–6. Finally run the plotting cell.


In [ ]:
# Run once with "plain", then change to "prefixed" and rerun
# this cell plus the experiment and plotting cells.
OUTPUT_FORMAT = "plain"  # "plain": 18; "prefixed": ### 18

if "_QWEN_EXPERIMENT_BASE_OUT" not in globals():
    _QWEN_EXPERIMENT_BASE_OUT = OUT

OUT = _QWEN_EXPERIMENT_BASE_OUT / OUTPUT_FORMAT
OUT.mkdir(parents=True, exist_ok=True)

BASE_INSTRUCTION = (
    "Solve the arithmetic problem. "
)

FORMAT_INSTRUCTIONS = {
    "plain": (
        "In the final response, output the final answer alone. No extra text."
    ),
    "prefixed": (
        "In the final response, output the literal prefix '### ' "
        "immediately followed by the final answer. No extra text."
    ),
}

END_THINK = tokenizer.convert_tokens_to_ids("</think>")

if (
    not isinstance(END_THINK, int)
    or END_THINK == tokenizer.unk_token_id
):
    raise RuntimeError("Cannot identify Qwen's </think> token")


def decode_output(token_ids):
    return tokenizer.decode(
        token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )


def generate_one(question, mode):
    text = (
        BASE_INSTRUCTION
        + " "
        + FORMAT_INSTRUCTIONS[OUTPUT_FORMAT]
        + "\n\nProblem: "
        + question
    )

    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=(mode == "CoT"),
        return_tensors="pt",
    ).to(model.device)

    sampling = (
        dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)
        if mode == "CoT"
        else dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
    )

    with torch.inference_mode():
        full = model.generate(
            ids,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            **sampling,
        )[0]

    generated = full[ids.shape[1]:].tolist()

    if len(generated) >= MAX_NEW_TOKENS:
        return None

    end = [
        i for i, token in enumerate(generated)
        if token == END_THINK
    ]

    if mode == "CoT" and not end:
        return None
    if mode == "Direct" and end:
        return None

    answer_start = end[-1] + 1 if end else 0
    think_ids = generated[:end[-1]] if end else []
    think = decode_output(think_ids)

    final_ids = generated[answer_start:]
    raw_final = decode_output(final_ids)
    final = raw_final.strip()

    pattern = (
        r"(?P<number>\d+)"
        if OUTPUT_FORMAT == "plain"
        else r"### (?P<number>\d+)"
    )
    match = re.fullmatch(pattern, final)

    if match is None:
        return None

    # Find the actual generated token containing the first answer digit.
    # A space and the numeral may belong to the same Qwen token.
    digit_char = (
        len(raw_final)
        - len(raw_final.lstrip())
        + match.start("number")
    )

    first = None

    for j in range(len(final_ids)):
        before = decode_output(final_ids[:j])
        after = decode_output(final_ids[:j + 1])

        if len(after) > digit_char:
            new_text = after[len(before):]

            # No separate pre-answer state if ### and the
            # answer digit occur in the same token.
            if not re.match(r"^\s*\d", new_text):
                return None

            first = j
            break

    if first is None:
        return None

    anchor = ids.shape[1] + answer_start + first - 1

    if anchor < 0:
        return None

    return dict(
        mode=mode,
        answer_format=OUTPUT_FORMAT,
        prompt_ids=ids[0].tolist(),
        generated_ids=generated,
        think=think,
        answer=canonical(match.group("number")),
        anchor=anchor,
        full_ids=full[:anchor + 1].unsqueeze(0),
        final=final,
    )

In [ ]:
# Trace exactly the selected positions. In a causal decoder, layer output at
# position p consumes token p and predicts token p+1. 
@torch.no_grad()
def trace_positions(run):
    anchor=run["anchor"]
    positions=list(range(max(0,anchor-LOOKBACK),anchor+1))
    recorded={}
    handles=[]
    for layer in sorted(set(layers)|{lens_model.n_layers-1}):
        def hook(_,inputs,output,layer=layer):
            h=output[0] if isinstance(output,tuple) else output
            recorded[layer]=h[0,positions,:].detach().to("cpu",dtype=torch.float32)
        handles.append(lens_model.layers[layer].register_forward_hook(hook))
    try:
        model(run["full_ids"].to(model.device),use_cache=False)
    finally:
        for h in handles: h.remove()
    assert set(recorded)==set(layers)|{lens_model.n_layers-1}
    ids=run["prompt_ids"]+run["generated_ids"]
    return positions,recorded,ids

def rank_and_spec(logits, true_id, wrong_ids):
    target=logits[:,true_id]
    rank=1+(logits>target[:,None]).sum(-1)
    spec=(target[:,None]>logits[:,wrong_ids]).float().mean(-1)
    return (-torch.log10(rank.float())).cpu().numpy(), spec.cpu().numpy()

@torch.no_grad()
def measure_run(run,items,dataset_idx):
    positions,trace,ids=trace_positions(run)
    rows=[]; audit=[]
    y_first_token_id=ids[run["anchor"]+1]
    for s,p in enumerate(positions):
        audit.append(dict(dataset_idx=dataset_idx,mode=run["mode"],x=p-run["anchor"],
                          state_token=tokenizer.decode([ids[p]],skip_special_tokens=False),
                          next_token=tokenizer.decode([ids[p+1]],skip_special_tokens=False),
                          is_generated_state=p>=len(run["prompt_ids"])))
    final_h=trace[lens_model.n_layers-1].to(model.device)
    lm_logits=lens_model.unembed(final_h)
    lm_y_target=lm_logits[:,y_first_token_id]
    lm_y_rank=1+(lm_logits>lm_y_target[:,None]).sum(-1)
    lm_y_v=(-torch.log10(lm_y_rank.float())).cpu().numpy()
    for layer in layers:
        h=trace[layer].to(model.device)
        # Saved J matrices are fp16; cast per layer so matmul uses fp32 on both sides.
        J=lens.jacobians[layer].to(device=model.device,dtype=torch.float32)
        logits=lens_model.unembed(h @ J.T)
        y_target=logits[:,y_first_token_id]
        y_rank=1+(logits>y_target[:,None]).sum(-1)
        y_v=(-torch.log10(y_rank.float())).cpu().numpy()
        for z_idx,z,correct_id,controls in items:
            v,spec=rank_and_spec(logits,correct_id,controls)
            lm_v,_=rank_and_spec(lm_logits,correct_id,controls)
            for j,p in enumerate(positions):
                rows.append(dict(dataset_idx=dataset_idx,z_idx=z_idx,Z=z,mode=run["mode"],
                                 layer=layer,x=p-run["anchor"],visibility=float(v[j]),
                                 specificity=float(spec[j]),lm_visibility=float(lm_v[j]),
                                 y_first_token_visibility=float(y_v[j]),
                                 y_first_token_lm_visibility=float(lm_y_v[j])))
        del logits,J
    return rows,audit


In [ ]:
# Same-problem, same-Z Direct/CoT cohort. Files are written in each iteration
all_rows=[]; audit_rows=[]; run_rows=[]; excluded=Counter()
scan=data if MAX_SCAN is None else data[:MAX_SCAN]
for idx,sample in enumerate(scan):
    gold=canonical(sample["answer"])
    if gold is None or not gold.isdigit(): excluded["unsupported_gold"]+=1; continue
    runs={mode:generate_one(sample["question"],mode) for mode in ("Direct","CoT")}
    if any(r is None for r in runs.values()): excluded["invalid_output_or_missing_think"]+=1; continue
    if any(r["answer"]!=gold for r in runs.values()): excluded["either_wrong"]+=1; continue
    zs=gold_z_values(sample)
    if REQUIRE_Z_WRITTEN_IN_COT:
        zs=[z for z in zs if z in [canonical(m.group()) for m in NUMBER_RE.finditer(runs["CoT"]["think"])]]
    if not zs: excluded["no_gold_Z_written_in_CoT"]+=1; continue
    forbidden={gold}|set(zs)
    for text in (sample["question"],runs["CoT"]["think"]):
        forbidden|={canonical(m.group()) for m in NUMBER_RE.finditer(text)}
    items=[]
    for z_idx,z in enumerate(zs):
        controls=wrong_controls(z,forbidden)
        if len(controls)<MIN_WRONG: continue
        items.append((z_idx,z,token_id(z),[token_id(v) for v in controls]))
    if not items: excluded["no_controls"]+=1; continue
    for mode in ("Direct","CoT"):
        rows,audit=measure_run(runs[mode],items,idx)
        all_rows.extend(rows);audit_rows.extend(audit)
        run_rows.append(dict(dataset_idx=idx,mode=mode,gold=gold,final=runs[mode]["final"],
                             think=runs[mode]["think"],n_Z=len(items)))
    print(f"\rScanned {idx+1}/{len(scan)}, retained={len(run_rows)//2}",end="",flush=True)
print("\nExclusions",dict(excluded))
if not all_rows: raise RuntimeError("No valid paired observations: inspect outputs and the Z filter")
layer_z=pd.DataFrame(all_rows)
layer_problem=layer_z.groupby(["dataset_idx","mode","layer","x"],as_index=False)[["visibility","specificity","lm_visibility","y_first_token_visibility","y_first_token_lm_visibility"]].mean()
global_z=layer_z.groupby(["dataset_idx","z_idx","Z","mode","x"],as_index=False)[["visibility","specificity","lm_visibility","y_first_token_visibility","y_first_token_lm_visibility"]].mean()
global_problem=global_z.groupby(["dataset_idx","mode","x"],as_index=False)[["visibility","specificity","lm_visibility","y_first_token_visibility","y_first_token_lm_visibility"]].mean()
for name,frame in [("layer_Z",layer_z),("layer_problem",layer_problem),
                   ("global_Z",global_z),("global_problem",global_problem),
                   ("alignment",pd.DataFrame(audit_rows)),("runs",pd.DataFrame(run_rows))]:
    frame.to_csv(OUT/f"qwen3_{name}.csv",index=False)
print("Retained paired problems",global_problem.dataset_idx.nunique(),"Z",layer_z[["dataset_idx","z_idx"]].drop_duplicates().shape[0])
display(pd.DataFrame(audit_rows).query("x >= -3").groupby(["mode","x","next_token"]).size().rename("n").reset_index().head(30))


In [ ]:
# Source-layer profiles for both formats and targets.
# Requires the experiment cell to have been run once with each OUTPUT_FORMAT.
# No generation or J-Lens readout is repeated here.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

BASE_OUT = Path(_QWEN_EXPERIMENT_BASE_OUT)
FORMATS = {"plain": [0], "prefixed": [-2, -1, 0]}
TARGETS = {
    "Z": ("visibility", r"Visibility $-\log_{10}\mathrm{rank}(Z)$"),
    "Y": ("y_first_token_visibility", r"Visibility $-\log_{10}\mathrm{rank}(Y)$"),
}
COLORS = {"Direct": "#4477AA", "CoT": "#EE7733"}
MARKERS = {"Direct": "o", "CoT": "s"}
N_BOOT = 3000

PAPER_STYLE = {
    "font.family": "DejaVu Sans", "font.size": 8,
    "axes.labelsize": 8.5, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "legend.fontsize": 8,
    "axes.linewidth": 0.65, "pdf.fonttype": 42,
}

def panel_labels(audit, fmt, positions):
    # A state at x predicts the token stored in audit.next_token at x.
    if fmt == "plain":
        return [r"(a) Predict $Y$"]
    names = []
    expected = {-2: "###", -1: " "}
    for j, x in enumerate(positions):
        if x == 0:
            names.append(f"({chr(97+j)}) Predict $Y$")
            continue
        tokens = audit.loc[audit.x.eq(x), "next_token"].dropna().astype(str)
        if len(tokens) and tokens.eq(expected[x]).all():
            label = "Predict ###" if x == -2 else "Predict space"
        else:
            label = rf"$x={x}$"
            print(f"prefixed: x={x} tokens differ from the expected prefix; "
                  f"using relative position. Counts: {tokens.value_counts().head(8).to_dict()}")
        names.append(f"({chr(97+j)}) {label}")
    return names

def plot_final_layer_profiles(frame, audit, fmt, target, metric, ylabel):
    positions = FORMATS[fmt]
    layers = np.sort(frame.layer.dropna().unique().astype(int))
    if not len(layers):
        raise ValueError(f"No source layers in {fmt} data")
    labels = panel_labels(audit, fmt, positions)
    width = 3.4 if fmt == "plain" else 7.2
    fig, axes = plt.subplots(1, len(positions), figsize=(width, 2.65),
                             sharey=True, squeeze=False)
    axes = axes[0]
    fig.subplots_adjust(left=0.17 if fmt == "plain" else 0.09,
                        right=0.99, bottom=0.30, top=0.82,
                        wspace=0.17)
    ticks = layers[np.unique(np.linspace(
        0, len(layers)-1, min(6, len(layers)))
        .round().astype(int))]

    for j, (ax, x) in enumerate(zip(axes, positions)):
        for mode in ("Direct", "CoT"):
            wide = (frame.loc[frame.x.eq(x) & frame["mode"].eq(mode)]
                    .pivot(index="dataset_idx", columns="layer", values=metric)
                    .reindex(columns=layers))
            values = wide.to_numpy(float)
            if len(values) < 2:
                raise ValueError(f"{fmt}, {target}, {mode}, x={x}: "
                                 "fewer than two retained problems")
            rng = np.random.default_rng(SEED)
            boot = np.stack([
                np.nanmean(values[rng.integers(len(values), size=len(values))], axis=0)
                for _ in range(N_BOOT)
            ])
            lo, hi = np.nanpercentile(boot, [2.5, 97.5], axis=0)
            ax.fill_between(layers, lo, hi, color=COLORS[mode],
                            alpha=0.15, linewidth=0)
            ax.plot(layers, np.nanmean(values, axis=0),
                    color=COLORS[mode], linewidth=1.55,
                    marker=MARKERS[mode], markersize=2.8,
                    markeredgecolor="white", markeredgewidth=0.3)
        ax.set_xticks(ticks)
        ax.set_xlim(layers[0]-0.3, layers[-1]+0.3)
        ax.grid(axis="y", color="0.90", linewidth=0.45)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.text(0.5, -0.24, labels[j], ha="center", va="top",
                transform=ax.transAxes, fontsize=8)

    axes[0].set_ylabel(ylabel)
    fig.legend(handles=[Line2D([0], [0], color=COLORS[mode],
                               marker=MARKERS[mode], lw=1.55,
                               label=mode) for mode in ("Direct", "CoT")],
               loc="upper center", bbox_to_anchor=(0.55, 1.0),
               ncol=2, frameon=False)
    fig.supxlabel("Source layer", y=0.025, fontsize=8.5)
    stem = f"paper_qwen3_{fmt}_{target}_final_layer_lines"
    fig.savefig(BASE_OUT / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.025)
    fig.savefig(BASE_OUT / f"{stem}.png", dpi=450,
                bbox_inches="tight", pad_inches=0.025)
    plt.show()
    plt.close(fig)

with plt.rc_context(PAPER_STYLE):
    for fmt in FORMATS:
        directory = BASE_OUT / fmt
        frame = pd.read_csv(directory / "qwen3_layer_problem.csv")
        audit = pd.read_csv(directory / "qwen3_alignment.csv",
                            keep_default_na=False)
        for target, (metric, ylabel) in TARGETS.items():
            plot_final_layer_profiles(frame, audit, fmt, target, metric, ylabel)
print("Saved four figures: plain/prefixed × Z/Y (PDF and PNG).")


## Interpretation
The plain format contributes one answer-token prediction panel. The prefixed format contributes three panels at x = -2, -1, 0; the saved alignment audit determines whether the first two can be labeled as predictions of `###` and a space. The x = 0 panel predicts the Y answer token. All plots use the same correctly answered Direct/CoT problems within each format and show means and pointwise problem-bootstrap 95% intervals at each source layer. Cohorts can differ across formats.
